In [1]:
from pathlib import Path
import polars as pl

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"

train = pl.read_csv(
    DATA_DIR / "train.csv",
    try_parse_dates=True,
)

print("Rows and columns:", train.shape)
print("Column types:", train.schema)

train.head()

Rows and columns: (3000888, 6)
Column types: Schema({'id': Int64, 'date': Date, 'store_nbr': Int64, 'family': String, 'sales': Float64, 'onpromotion': Int64})


id,date,store_nbr,family,sales,onpromotion
i64,date,i64,str,f64,i64
0,2013-01-01,1,"""AUTOMOTIVE""",0.0,0
1,2013-01-01,1,"""BABY CARE""",0.0,0
2,2013-01-01,1,"""BEAUTY""",0.0,0
3,2013-01-01,1,"""BEVERAGES""",0.0,0
4,2013-01-01,1,"""BOOKS""",0.0,0


### Missing and invalid values

In [2]:
print("Missing values:")
display(train.null_count())

display(train.select(
    (pl.col("sales") < 0).sum().alias("negative_sales"),
    (~pl.col("sales").is_finite()).sum().alias("nonfinite_sales"),
    (pl.col("onpromotion") < 0).sum().alias("negative_promotions"),
))


Missing values:


id,date,store_nbr,family,sales,onpromotion
u32,u32,u32,u32,u32,u32
0,0,0,0,0,0


negative_sales,nonfinite_sales,negative_promotions
u32,u32,u32
0,0,0


### Duplicate records

In [3]:
display(train.select(
    pl.col("id").is_duplicated().sum().alias("duplicate_id_rows"),
    pl.struct(["date", "store_nbr", "family"]).is_duplicated().sum().alias("duplicate_key_rows"),
))


duplicate_id_rows,duplicate_key_rows
u32,u32
0,0


### Date coverage

In [4]:
print("Date range:", train["date"].min(), "to", train["date"].max())
print("Observed days:", train["date"].n_unique())

calendar = pl.DataFrame({"date": pl.date_range(
    train["date"].min(), train["date"].max(), eager=True
)})
missing_dates = calendar.join(train.select("date").unique(), on="date", how="anti").sort("date")
print("Missing calendar days:", missing_dates.height)
display(missing_dates)


Date range: 2013-01-01 to 2017-08-15
Observed days: 1684
Missing calendar days: 4


date
date
2013-12-25
2014-12-25
2015-12-25
2016-12-25


### Brief conclusion
No modification to the training data is needed based on these basic checks: there are no missing values, duplicate IDs or daily store-family keys, negative sales or promotions, or nonfinite sales.

Four calendar dates are absent: December 25 in 2013, 2014, 2015, and 2016.